# Federated 3D U-Net training (BraTS-PEDs) — Colab launcher

**One-time setup:** this repo is private, so cloning needs a GitHub token.
1. Create a fine-grained personal access token at github.com → Settings → Developer settings
   → Personal access tokens, scoped to just this repo, with `Contents: Read-only`.
2. In Colab, click the key icon (🔑) in the left sidebar → **Secrets** → add a new secret
   named `GITHUB_TOKEN` with that token as the value, and toggle "Notebook access" on.
3. Never paste the token directly into a cell — it gets saved into notebook/checkpoint files
   and is easy to accidentally leak (e.g. in a screenshot or shared notebook). The Secrets
   panel keeps it out of the notebook entirely.

Then fill in `CACHE_PATH` below (the Drive folder with `<subject_id>.npz` files) and run all cells.

Checkpoints save to Drive (`CHECKPOINT_DIR`), not the Colab VM's local disk — so a session
disconnect (which Colab does randomly) doesn't lose progress. Re-running this notebook after a
disconnect will resume automatically from the last saved epoch/round.

This starts with the **baseline** config (no augmentation/federation/domain-adaptation) — see
`00_shared/CONTRACTS.md` for the full ablation matrix (baseline → +aug → +fed → +DA) that
section 03 will eventually run and score.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from google.colab import userdata

GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')  # from the Secrets panel (key icon), never typed here

!git clone -q -b federated-brain-tumor-segmentation https://{GITHUB_TOKEN}@github.com/Sanad-Manga/Pediatric-Brain-Tumor-Model.git /content/repo
%cd /content/repo/01_model_federated
!pip install -q -r requirements.txt

del GITHUB_TOKEN  # don't leave the token sitting in a notebook variable longer than needed

In [ ]:
# --- fill these in ---
CACHE_PATH = "/content/drive/MyDrive/<path-to-cache_96cube-in-your-Drive>"
CHECKPOINT_DIR = "/content/drive/MyDrive/pediatric-brain-tumor-checkpoints"

MANIFEST_DIR = "/content/repo/00_shared/manifests"
HOSPITAL_A = f"{MANIFEST_DIR}/hospitalA.json"
HOSPITAL_B = f"{MANIFEST_DIR}/hospitalB.json"
HELDOUT = f"{MANIFEST_DIR}/heldout.json"

In [ ]:
import sys
sys.path.insert(0, "/content/repo/01_model_federated")

import torch
print("CUDA available:", torch.cuda.is_available())

from src.config import TrainConfig
from src.federated import train_federated
from src.train_single import train_single_client

## Baseline: single-client sanity check
Small run on Hospital A only, mostly to confirm the real cache loads and one real epoch's
timing on this GPU before committing to a long federated run.

In [ ]:
sanity_config = TrainConfig(
    data_mode="real",
    cache_path=CACHE_PATH,
    run_id="baseline_sanity",
    checkpoint_dir=CHECKPOINT_DIR,
)
model, losses = train_single_client(sanity_config, HOSPITAL_A, num_epochs=2, resume=True)
print(losses)

## Baseline: full FedAvg run across Hospital A + Hospital B
Adjust `NUM_ROUNDS` / `LOCAL_EPOCHS` once you've seen the per-epoch timing above.
`resume=True` means re-running this cell after a disconnect continues from the last saved round.

In [ ]:
NUM_ROUNDS = 20
LOCAL_EPOCHS = 1

fed_config = TrainConfig(
    use_federation=True,
    data_mode="real",
    cache_path=CACHE_PATH,
    run_id="baseline_fedavg",
    checkpoint_dir=CHECKPOINT_DIR,
)
global_model, round_losses = train_federated(
    fed_config,
    client_manifest_paths=[HOSPITAL_A, HOSPITAL_B],
    num_rounds=NUM_ROUNDS,
    local_epochs=LOCAL_EPOCHS,
    resume=True,
)
print(round_losses)

## Next steps (not built in this section)
- Evaluating Dice on `HELDOUT` and writing the ablation-matrix CSV is section 03's job
  (`00_shared/CONTRACTS.md` results schema).
- To try other ablation flags, change `use_augmentation` / `use_domain_adaptation` on
  `fed_config` above once those sections have something real to plug in — the flags already
  work, they're just no-ops until then.